# Solution A: Final Test Predictions for NLI

## 1. Import libraries

In [1]:
import re
from pathlib import Path

import pandas as pd
from scipy.sparse import hstack, csr_matrix
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics.pairwise import cosine_similarity

## 2. Load training and test data

In [2]:
train_df = pd.read_csv("../data/raw/train.csv")
test_df = pd.read_csv("../data/raw/test.csv")

print("Training set shape:", train_df.shape)
print("Test set shape:", test_df.shape)

display(train_df.head())
display(test_df.head())

Training set shape: (24432, 3)
Test set shape: (3302, 2)


,premise,hypothesis,label
0,yeah i don't know cut California in half or so...,Yeah. I'm not sure how to make that fit. Maybe...,1
1,actual names will not be used,"For the sake of privacy, actual names are not ...",1
2,The film was directed by Randall Wallace.,The film was directed by Randall Wallace and s...,1
3,"""How d'you know he'll sign me on?""Anse studie...",Anse looked at himself in a cracked mirror.,1
4,In the light of the candles his cheeks looked ...,Drew regarded his best friend and noted that i...,1


,premise,hypothesis
0,"Boy wearing red hat, blue jacket pushing plow ...",The boy is surrounded by snow
1,A blond woman in a black shirt is standing beh...,The woman is standing.
2,Three people in uniform are outdoors and are o...,Uniformed people are outside
3,"A person, in a striped blue shirt and pants, i...",The person is running
4,"A man, woman, and child get their picture take...",A family on vacation is posing.


## 3. Build a shared TF-IDF vocabulary

In [3]:
shared_vectorizer = TfidfVectorizer(
    max_features=30000,
    ngram_range=(1, 2),
    sublinear_tf=True,
    min_df=2
)

shared_training_corpus = pd.concat(
    [
        train_df["premise"].astype(str),
        train_df["hypothesis"].astype(str),
    ],
    axis=0,
)

shared_vectorizer.fit(shared_training_corpus)

X_train_premise = shared_vectorizer.transform(train_df["premise"].astype(str))
X_train_hypothesis = shared_vectorizer.transform(train_df["hypothesis"].astype(str))

X_test_premise = shared_vectorizer.transform(test_df["premise"].astype(str))
X_test_hypothesis = shared_vectorizer.transform(test_df["hypothesis"].astype(str))

print("Train premise TF-IDF shape:", X_train_premise.shape)
print("Train hypothesis TF-IDF shape:", X_train_hypothesis.shape)
print("Test premise TF-IDF shape:", X_test_premise.shape)
print("Test hypothesis TF-IDF shape:", X_test_hypothesis.shape)

Train premise TF-IDF shape: (24432, 30000)
Train hypothesis TF-IDF shape: (24432, 30000)
Test premise TF-IDF shape: (3302, 30000)
Test hypothesis TF-IDF shape: (3302, 30000)


## 4. Construct vector interaction features

In [4]:
X_train_abs_diff = abs(X_train_premise - X_train_hypothesis)
X_test_abs_diff = abs(X_test_premise - X_test_hypothesis)

X_train_elementwise_product = X_train_premise.multiply(X_train_hypothesis)
X_test_elementwise_product = X_test_premise.multiply(X_test_hypothesis)

## 5. Define helper functions for lexical feature extraction

In [5]:
NEGATION_WORDS = {"not", "no", "never", "none", "nothing"}

def tokenize(text: str) -> list[str]:
    text = str(text).lower()
    return re.findall(r"\b\w+\b|n't", text)

def token_set(text: str) -> set[str]:
    return set(tokenize(text))

def jaccard_overlap(row: pd.Series) -> float:
    premise_tokens = token_set(row["premise"])
    hypothesis_tokens = token_set(row["hypothesis"])
    union = premise_tokens | hypothesis_tokens
    if not union:
        return 0.0
    return len(premise_tokens & hypothesis_tokens) / len(union)

def shared_token_count(row: pd.Series) -> int:
    premise_tokens = token_set(row["premise"])
    hypothesis_tokens = token_set(row["hypothesis"])
    return len(premise_tokens & hypothesis_tokens)

def premise_covers_hypothesis(row: pd.Series) -> float:
    premise_tokens = token_set(row["premise"])
    hypothesis_tokens = token_set(row["hypothesis"])
    if len(hypothesis_tokens) == 0:
        return 0.0
    return len(premise_tokens & hypothesis_tokens) / len(hypothesis_tokens)

def hypothesis_covers_premise(row: pd.Series) -> float:
    premise_tokens = token_set(row["premise"])
    hypothesis_tokens = token_set(row["hypothesis"])
    if len(premise_tokens) == 0:
        return 0.0
    return len(premise_tokens & hypothesis_tokens) / len(premise_tokens)

def negation_count(text: str) -> int:
    tokens = tokenize(text)
    return sum(1 for token in tokens if token in NEGATION_WORDS or token == "n't")

## 6. Construct handcrafted pairwise features

In [6]:
train_cosine = cosine_similarity(X_train_premise, X_train_hypothesis).diagonal()
test_cosine = cosine_similarity(X_test_premise, X_test_hypothesis).diagonal()

train_features = pd.DataFrame(
    {
        "premise_char_length": train_df["premise"].astype(str).str.len(),
        "hypothesis_char_length": train_df["hypothesis"].astype(str).str.len(),
        "char_length_difference": (
            train_df["premise"].astype(str).str.len()
            - train_df["hypothesis"].astype(str).str.len()
        ).abs(),
        "jaccard_overlap": train_df.apply(jaccard_overlap, axis=1),
        "shared_token_count": train_df.apply(shared_token_count, axis=1),
        "premise_covers_hypothesis": train_df.apply(premise_covers_hypothesis, axis=1),
        "hypothesis_covers_premise": train_df.apply(hypothesis_covers_premise, axis=1),
        "premise_negation_count": train_df["premise"].astype(str).apply(negation_count),
        "hypothesis_negation_count": train_df["hypothesis"].astype(str).apply(negation_count),
        "tfidf_cosine_similarity": train_cosine,
    }
)

train_features["negation_count_difference"] = (
    train_features["premise_negation_count"] - train_features["hypothesis_negation_count"]
).abs()

test_features = pd.DataFrame(
    {
        "premise_char_length": test_df["premise"].astype(str).str.len(),
        "hypothesis_char_length": test_df["hypothesis"].astype(str).str.len(),
        "char_length_difference": (
            test_df["premise"].astype(str).str.len()
            - test_df["hypothesis"].astype(str).str.len()
        ).abs(),
        "jaccard_overlap": test_df.apply(jaccard_overlap, axis=1),
        "shared_token_count": test_df.apply(shared_token_count, axis=1),
        "premise_covers_hypothesis": test_df.apply(premise_covers_hypothesis, axis=1),
        "hypothesis_covers_premise": test_df.apply(hypothesis_covers_premise, axis=1),
        "premise_negation_count": test_df["premise"].astype(str).apply(negation_count),
        "hypothesis_negation_count": test_df["hypothesis"].astype(str).apply(negation_count),
        "tfidf_cosine_similarity": test_cosine,
    }
)

test_features["negation_count_difference"] = (
    test_features["premise_negation_count"] - test_features["hypothesis_negation_count"]
).abs()

display(train_features.head())
display(test_features.head())

,premise_char_length,hypothesis_char_length,char_length_difference,jaccard_overlap,shared_token_count,premise_covers_hypothesis,hypothesis_covers_premise,premise_negation_count,hypothesis_negation_count,tfidf_cosine_similarity,negation_count_difference
0,53,113,60,0.280000,7,0.333333,0.636364,0,1,0.302948,1
1,29,51,22,0.333333,4,0.400000,0.666667,1,1,0.355069,0
2,41,62,21,0.636364,7,0.636364,1.000000,0,0,0.719915,0
3,122,44,78,0.107143,3,0.375000,0.130435,0,0,0.194861,0
4,185,98,87,0.195122,8,0.500000,0.242424,1,0,0.182190,1


,premise_char_length,hypothesis_char_length,char_length_difference,jaccard_overlap,shared_token_count,premise_covers_hypothesis,hypothesis_covers_premise,premise_negation_count,hypothesis_negation_count,tfidf_cosine_similarity,negation_count_difference
0,54,29,25,0.142857,2,0.333333,0.200000,0,0,0.174134,0
1,60,22,38,0.272727,3,0.750000,0.300000,0,0,0.319820,0
2,91,28,63,0.111111,2,0.500000,0.125000,0,0,0.072334,0
3,62,21,41,0.250000,3,0.750000,0.272727,0,0,0.365089,0
4,74,31,43,0.052632,1,0.166667,0.071429,0,0,0.000000,0


## 7. Combine all feature blocks

In [7]:
X_train_handcrafted = csr_matrix(train_features.values)
X_test_handcrafted = csr_matrix(test_features.values)

X_train = hstack(
    [
        X_train_premise,
        X_train_hypothesis,
        X_train_abs_diff,
        X_train_elementwise_product,
        X_train_handcrafted,
    ]
)

X_test = hstack(
    [
        X_test_premise,
        X_test_hypothesis,
        X_test_abs_diff,
        X_test_elementwise_product,
        X_test_handcrafted,
    ]
)

y_train = train_df["label"].values

print("Final training matrix shape:", X_train.shape)
print("Final test matrix shape:", X_test.shape)

Final training matrix shape: (24432, 120011)
Final test matrix shape: (3302, 120011)


## 8. Train the final logistic regression model

In [8]:
final_a_model = LogisticRegression(
    max_iter=3000,
    C=0.25,
    solver="liblinear"
)

final_a_model.fit(X_train, y_train)
test_predictions = final_a_model.predict(X_test)

## 9. Save submission predictions

In [9]:
output_path = Path("../outputs/predictions/Group_56_A.csv")
output_path.parent.mkdir(parents=True, exist_ok=True)

submission_df = pd.DataFrame({"prediction": test_predictions})
submission_df.to_csv(output_path, index=False)

print(f"Saved submission file to: {output_path}")
display(submission_df.head())

Saved submission file to: ../outputs/predictions/Group_56_A.csv


,prediction
0,1
1,0
2,1
3,0
4,1


## 10. Sanity checks

In [10]:
print("Submission shape:", submission_df.shape)
print("Submission columns:", submission_df.columns.tolist())
print("Unique prediction values:", sorted(submission_df["prediction"].unique().tolist()))
print("Test rows:", len(test_df))
print("Submission rows:", len(submission_df))

Submission shape: (3302, 1)
Submission columns: ['prediction']
Unique prediction values: [0, 1]
Test rows: 3302
Submission rows: 3302
